## Import 


In [1]:
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType, IntegerType
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml import Pipeline
 
print("=" * 60)
print("SILVER CLEANING — Credit Card Fraud Detection")
print("=" * 60)


StatementMeta(, 76037853-25fd-4f12-8a4b-3013a5f937fe, 3, Finished, Available, Finished, False)

SILVER CLEANING — Credit Card Fraud Detection


## Load Bronze Data

In [2]:
df = spark.read.format("delta").table("bronze_transactions")
total = df.count()
print(f"Total transactions : {total:,}")      # 284,807
print(f"Columns            : {len(df.columns)}")  # 31
 
# Target distribution — critical to check
df.groupBy("Class").count().orderBy("Class").show()
pos_rate = df.filter(F.col("Class") == 1).count() / total
print(f"Fraud rate         : {pos_rate:.4%}")   # 0.1727%
print(f"Imbalance ratio    : 1:{int(1/pos_rate)}")  # 1:578

StatementMeta(, 76037853-25fd-4f12-8a4b-3013a5f937fe, 4, Finished, Available, Finished, False)

Total transactions : 284,807
Columns            : 31
+-----+------+
|Class| count|
+-----+------+
|    0|284315|
|    1|   492|
+-----+------+

Fraud rate         : 0.1727%
Imbalance ratio    : 1:578


## Cast all V features to FloatType

In [3]:
v_cols = [f"V{i}" for i in range(1, 29)]
for c in v_cols:
    df = df.withColumn(c, F.col(c).cast(FloatType()))
 
df = (df
    .withColumn("Amount", F.col("Amount").cast(FloatType()))
    .withColumn("Time",   F.col("Time").cast(FloatType()))
    .withColumn("Class",  F.col("Class").cast(IntegerType()))
)

StatementMeta(, 76037853-25fd-4f12-8a4b-3013a5f937fe, 5, Finished, Available, Finished, False)

## Extract hour-of-day from Time

In [4]:
# Time is seconds elapsed since first transaction.
# Fraud spikes at night (hours 0–6) — this is a strong feature.
df = df.withColumn("hour_of_day",
    (F.col("Time") / 3600).cast(IntegerType()) % 24
)
 
print("\nFraud rate by hour (top 5 fraud hours):")
df.groupBy("hour_of_day") \
  .agg(F.sum("Class").alias("fraud_count"),
       F.count("*").alias("total"),
       F.avg("Class").alias("fraud_rate")) \
  .orderBy("fraud_rate", ascending=False) \
  .show(5)

StatementMeta(, 76037853-25fd-4f12-8a4b-3013a5f937fe, 6, Finished, Available, Finished, False)


Fraud rate by hour (top 5 fraud hours):
+-----------+-----------+-----+--------------------+
|hour_of_day|fraud_count|total|          fraud_rate|
+-----------+-----------+-----+--------------------+
|          2|         57| 3328|0.017127403846153848|
|          4|         23| 2209| 0.01041195110909914|
|          3|         17| 3492|0.004868270332187858|
|          5|         11| 2990|0.003678929765886...|
|          7|         23| 7243|0.003175479773574486|
+-----------+-----------+-----+--------------------+
only showing top 5 rows



##  Amount bins

In [5]:
# Fraud often appears at specific amount ranges.
# New customers tested with tiny amounts first, then large.
df = df.withColumn("amount_bin",
    F.when(F.col("Amount") == 0,    "zero")
    .when(F.col("Amount") < 10,     "micro")
    .when(F.col("Amount") < 100,    "small")
    .when(F.col("Amount") < 1000,   "medium")
    .otherwise("large")
)
 
print("\nFraud rate by amount bin:")
df.groupBy("amount_bin") \
  .agg(F.sum("Class").alias("fraud"),
       F.count("*").alias("total"),
       F.avg("Class").alias("fraud_rate")) \
  .orderBy("fraud_rate", ascending=False) \
  .show()

StatementMeta(, 76037853-25fd-4f12-8a4b-3013a5f937fe, 7, Finished, Available, Finished, False)


Fraud rate by amount bin:
+----------+-----+------+--------------------+
|amount_bin|fraud| total|          fraud_rate|
+----------+-----+------+--------------------+
|      zero|   27|  1825|0.014794520547945205|
|     large|    9|  3069|0.002932551319648094|
|     micro|  222| 95489|0.002324875116505566|
|    medium|  121| 54316|0.002227704543780838|
|     small|  113|130108|8.685092384788022E-4|
+----------+-----+------+--------------------+



## Scale Amount and Time

In [6]:
# V1–V28 are already scaled (PCA output).
# Amount ranges $0–$25,691. Time ranges 0–172,792 seconds.
# Isolation Forest is distance-based — unscaled Amount would
# dominate the anomaly score. Must scale before IF training.
FEATURE_COLS = v_cols + ["Amount", "hour_of_day"]
 
assembler = VectorAssembler(
    inputCols=FEATURE_COLS, outputCol="features_raw",
    handleInvalid="skip"
)
scaler = StandardScaler(
    inputCol="features_raw", outputCol="features_scaled",
    withStd=True, withMean=True
)
 
pipeline = Pipeline(stages=[assembler, scaler])
pipeline_model = pipeline.fit(df)
df_scaled = pipeline_model.transform(df)
 
print(f"\nScaling complete. Features vector size: {len(FEATURE_COLS)}")

StatementMeta(, 76037853-25fd-4f12-8a4b-3013a5f937fe, 8, Finished, Available, Finished, False)


Scaling complete. Features vector size: 30


## Null audit

In [7]:
null_counts = [(c, df_scaled.filter(F.col(c).isNull()).count())
               for c in ["Amount","Time","Class","hour_of_day"] + v_cols[:5]]
null_counts.sort(key=lambda x: -x[1])
print("\nNull audit (key columns):")
for col_name, cnt in null_counts:
    flag = "⚠️ " if cnt > 0 else "✅"
    print(f"  {flag} {col_name:<20} {cnt:>6,}")

StatementMeta(, 76037853-25fd-4f12-8a4b-3013a5f937fe, 9, Finished, Available, Finished, False)


Null audit (key columns):
  ✅ Amount                    0
  ✅ Time                      0
  ✅ Class                     0
  ✅ hour_of_day               0
  ✅ V1                        0
  ✅ V2                        0
  ✅ V3                        0
  ✅ V4                        0
  ✅ V5                        0


## Write Silver Features


In [8]:
#  Write both the raw features and the scaled features vector
output_cols = v_cols + ["Amount","Time","Class","hour_of_day","amount_bin"]
 
df_out = df_scaled.select(output_cols)
df_out.write.format("delta").mode("overwrite") \
     .option("overwriteSchema", "true") \
     .saveAsTable("silver_features")
 
print(f"\n✅ silver_features: {df_out.count():,} rows, {len(output_cols)} cols")

StatementMeta(, 76037853-25fd-4f12-8a4b-3013a5f937fe, 10, Finished, Available, Finished, False)


✅ silver_features: 284,807 rows, 33 cols


In [9]:
display(df_out.limit(10))

StatementMeta(, 76037853-25fd-4f12-8a4b-3013a5f937fe, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6073061f-6c13-49a9-9fae-7b05074db7cc)